In [1]:
import matplotlib.pyplot as plt
from pathlib import Path
import numpy as np
import pandas as pd
import math
import seaborn as sns
import gc
import psutil
import os, random
import posixpath

from pyhdas.frequency import spectrogram, add_db, energy
from pyhdas.aggregate import quantile
from pyhdas.aragon import concat_raw_data, aragon_select_files

from datetime import timedelta, datetime

In [2]:
def compute_spectral_centroid(spectrogram_data):
    
    freq = spectrogram_data['freq'].values 
    Pxx = spectrogram_data['Pxx'].values  

    # Calculate spectral centroid for each time step
    numerator = np.sum(Pxx * freq[:, np.newaxis], axis=0)  
    denominator = np.sum(Pxx, axis=0)  
    spectral_centroid = numerator / denominator  

    return spectral_centroid

In [3]:
def compute_spectral_bandwidth(spectrogram_data, spectral_centroid):
    
    freq = spectrogram_data['freq'].values  # Frequencies (Hz)
    Pxx = spectrogram_data['Pxx'].values   # Power spectral density

    # Calculate Spectral Bandwidth
    numerator_bandwidth = np.sum(Pxx * ((freq[:, np.newaxis] - spectral_centroid)**2), axis=0)
    denominator_bandwidth = np.sum(Pxx, axis=0)
    spectral_bandwidth = np.sqrt(numerator_bandwidth / denominator_bandwidth)

    return spectral_bandwidth

In [6]:
def get_spectral_features():

    normal_data_dir = "data/19"
    output_dir = "plots/spectral_features_plots/4220"
    os.makedirs(output_dir, exist_ok=True)

    all_files = os.listdir(normal_data_dir)
    random.seed(12)
    selected_files = random.sample(all_files, 200, )  # pick 200 random files

    poi = 4220

    for file in selected_files:
        filepath = os.path.join(normal_data_dir, file)

        # Load and process data
        ds_raw = concat_raw_data([filepath])
        ds_raw = ds_raw.sel(position=poi)
        ds_spect = spectrogram(ds_raw, variable='strain')

        # Compute features
        spectral_centroid = compute_spectral_centroid(ds_spect)
        spectral_bandwidth = compute_spectral_bandwidth(ds_spect, spectral_centroid)

        # Plot spectral centroid and bandwidth
        plt.figure(figsize=(10, 5))
        plt.plot(spectral_centroid, label='Spectral Centroid',  color="royalblue")
        plt.plot(spectral_bandwidth, label='Spectral Bandwidth',  color="darkorange")
        plt.title(f'Spectral Features for {file}')
        plt.ylim((0, 500))
        plt.xlabel('Time Step')
        plt.ylabel('Frequency (Hz)')
        plt.legend()
        plt.grid(True, linestyle="--", alpha=0.6)
        plt.tight_layout()

        # Save plot
        save_path = os.path.join(output_dir, f'{os.path.splitext(file)[0]}_spectral_features.png')
        plt.savefig(save_path)
        plt.close()

    print(f"Saved spectral feature plots for {len(selected_files)} files to {output_dir}")


In [ ]:
get_spectral_features()

In [6]:
def get_sound_level_bands():

    normal_data_dir = "data/19"
    output_dir = "plots/sound_level_bands/4220"
    os.makedirs(output_dir, exist_ok=True)
    
    all_files = os.listdir(normal_data_dir)
    random.seed(12)
    selected_files = random.sample(all_files, 200)  # pick 200 random files

    freq_bands = [(0, 100), (50, 150), (100, 200), (150, 250), (200, 300)]
    poi = 4220

    for file in selected_files:
        filepath = os.path.join(normal_data_dir, file)

        ds_raw = concat_raw_data([filepath])
        ds_raw = ds_raw.sel(position=poi)
        ds_spect = spectrogram(ds_raw, variable='strain')

        # Total sound level
        ds_soundlevel = add_db(ds_spect[["Pxx"]].sum(dim="freq"))

        for low, high in freq_bands:
            # Band-specific sound level
            band = ds_spect[["Pxx"]].sel(freq=slice(low, high)).sum(dim="freq")
            band_db = add_db(band)

            # Plot both as time series
            plt.figure(figsize=(10, 4))
            plt.plot(ds_soundlevel.time, ds_soundlevel.Pxx_dB, label="Total Sound Level", color="black")
            plt.plot(band_db.time, band_db.Pxx_dB, label=f"{low}-{high} Hz Band", color="blue")

            plt.xlabel("Time")
            plt.ylabel("dB")
            plt.title(f"Sound Level vs {low}-{high} Hz Band\nFile: {file}")
            plt.legend()
            plt.tight_layout()

            # Save the plot
            plot_name = f"{os.path.splitext(file)[0]}_{low}_{high}.png"
            plt.savefig(os.path.join(output_dir, plot_name))
            plt.close()


In [7]:
get_sound_level_bands()

/var/lib/randd/.venv/lib64/python3.9/site-packages/pyhdas/frequency.py:139: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  assert new_dimension not in data.dims.keys(), (
/var/lib/randd/.venv/lib64/python3.9/site-packages/pyhdas/frequency.py:139: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  assert new_dimension not in data.dims.keys(), (
/var/lib/randd/.venv/lib64/python3.9/site-packages/pyhdas/frequency.py:139: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a map

In [2]:
# load the event table 
events = pd.read_csv("events_table.csv", parse_dates=["start", "end"])
freq_bands = [(0, 100), (50, 150), (100, 200)]

output_dir = "plots/sound_level_bands/4220_intrusions"
os.makedirs(output_dir, exist_ok=True)

for row, event in events.iterrows(): 

    start, end, poi, label = event["start"], event["end"], event["poi"], event["label_anon"]

    start = pd.Timestamp(start)
    end = pd.Timestamp(end)
    start_day = start.day

    end = end.tz_localize("UTC")
    start = start.tz_localize("UTC")

    end_file = start
    dir_data = Path(fr"data/{end.day}")


    while end_file < end:

        end_file = start + pd.Timedelta(seconds=60)

        print(start, end_file)

        # load the file based on the time provided
        file_list = list(aragon_select_files(dir_data, start, end_file, extension="bin"))
        if not file_list:
            print(f"The list is empty: {start}, {end}, {label}")
            continue

        # extract the sound levels 
        ds_raw = concat_raw_data(file_list)

        start = start.tz_localize(None)
        end_file = end_file.tz_localize(None)


        current_length = len(ds_raw.sel(time=slice(start, end_file), position=4220).time)

        # in case tehre are not enough data points, pad 
        if current_length < 120000:
            milliseconds_to_pad = int(math.ceil((120000 - current_length)/2))
            ds_raw = ds_raw.sel(time=slice(start, end_file+pd.Timedelta(milliseconds=milliseconds_to_pad)), position=4220)
        else:
            ds_raw = ds_raw.sel(time=slice(start, end_file), position=4220)
        

        ds_spect = spectrogram(ds_raw, variable='strain') 
        ds_soundlevel = ds_spect[["Pxx"]].sum(dim="freq")
        ds_soundlevel = add_db(ds_soundlevel) 

        for low, high in freq_bands:
        # Band-specific sound level
            band = ds_spect[["Pxx"]].sel(freq=slice(low, high)).sum(dim="freq")
            band_db = add_db(band)

            # Plot both as time series
            plt.figure(figsize=(10, 4))
            plt.plot(ds_soundlevel.time, ds_soundlevel.Pxx_dB, label="Total Sound Level", color="black")
            plt.plot(band_db.time, band_db.Pxx_dB, label=f"{low}-{high} Hz Band", color="blue")

            plt.xlabel("Time")
            plt.ylabel("dB")
            plt.title(f"Sound Level vs {low}-{high} Hz Band\nFile: {file_list}")
            plt.legend()
            plt.tight_layout()

            timestamp_str = start.strftime("%Y%m%dT%H%M%S")
            plot_name = f"{timestamp_str}_{poi}_{label}_{low}_{high}.png"

            plt.savefig(os.path.join(output_dir, plot_name))
            plt.close()
        
        end_file = end_file.tz_localize("UTC")

        start = end_file


    print(f"Done with row {row}")

2021-02-25 07:03:50+00:00 2021-02-25 07:04:50+00:00


/var/lib/randd/.venv/lib64/python3.9/site-packages/pyhdas/frequency.py:139: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  assert new_dimension not in data.dims.keys(), (


2021-02-25 07:04:50+00:00 2021-02-25 07:05:50+00:00


/var/lib/randd/.venv/lib64/python3.9/site-packages/pyhdas/frequency.py:139: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  assert new_dimension not in data.dims.keys(), (


2021-02-25 07:05:50+00:00 2021-02-25 07:06:50+00:00


...

/var/lib/randd/.venv/lib64/python3.9/site-packages/pyhdas/frequency.py:139: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  assert new_dimension not in data.dims.keys(), (


Done with row 64
2021-02-25 09:24:35+00:00 2021-02-25 09:25:35+00:00


/var/lib/randd/.venv/lib64/python3.9/site-packages/pyhdas/frequency.py:139: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  assert new_dimension not in data.dims.keys(), (


Done with row 65
2021-02-25 09:25:00+00:00 2021-02-25 09:26:00+00:00


SystemError: CPUDispatcher(<function denoise_with_reference_fiber at 0x7ff8e1ca61f0>) returned a result with an error set